<a href="https://colab.research.google.com/github/gilIolgenblum/CrowdingModeWorkshop/blob/main/tutorials/solutions/E2_fitting_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 2 — Solution: Fitting to Experimental Data

> ⚠️ **This is the solution notebook.** Students should attempt the exercises independently first.

In [ ]:
# (Only needed on Colab — skip if running locally)
# !pip install git+https://github.com/gilIolgenblum/CrowdingModeWorkshop.git

In [ ]:
import crowding as cr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

---
## Task B1 — Load and Inspect the Data

In [ ]:
import os
DATA_PATH = os.path.join(os.path.dirname(os.getcwd()), "fh_crowding_workshop", "app", "sample_data", "aq16_sucrose_binary_format1.csv")
# On Colab: DATA_PATH = "https://raw.githubusercontent.com/gilIolgenblum/CrowdingModeWorkshop/main/app/sample_data/aq16_sucrose_binary_format1.csv"

df = pd.read_csv(DATA_PATH)
print(f"Number of data points: {len(df)}")
print(df.head())

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(df["concentration"], df["dG"], label=r"$\Delta\Delta G^0$ exp")
ax.set_xlabel("Concentration (molal)")
ax.set_ylabel(r"$\Delta\Delta G^0$ [kJ/mol]")
ax.legend()
ax.set_title("AQ16 + Sucrose — Raw Data")
plt.tight_layout()
plt.show()

---
## Task B2 — Fit eps to ΔΔG⁰

In [ ]:
AQ16 = cr.Protein(SASA=242.6)
sucrose = cr.Cosolute(nu=11.9, chi=0.452, chiTS=-0.854)

model = cr.CrowdingModel(
    protein=AQ16, cosolute=sucrose,
    eps=0.0, epsTS=0.0,
    phiC_max=0.40, dphiC=0.001
)

exp_conc = df["concentration"].values
exp_ddG  = df["dG"].values

model.fit_eps(exp_conc, exp_ddG, concentration_type="molal")
print(f"Fitted eps = {model.eps:.4f} kJ/mol")

---
## Task B3 — Fit epsTS and Plot

In [ ]:
exp_ddH  = df["dH"].values
exp_TddS = df["TdS"].values

model.fit_epsTS(exp_conc, exp_ddH, concentration_type="molal")
print(f"Fitted epsTS = {model.epsTS:.4f} kJ/mol")

plotter = cr.Plotter(model)

fig1 = plotter.plot_ddG(concentration_type="molal", folding=False,
                        exp_conc=exp_conc, exp_ddG=exp_ddG)
fig1.suptitle(r"AQ16+Sucrose — $\Delta\Delta G^0$")
plt.show()

fig2 = plotter.plot_ddH(concentration_type="molal", folding=False,
                        exp_conc=exp_conc, exp_ddH=exp_ddH)
fig2.suptitle(r"AQ16+Sucrose — $\Delta\Delta H^0$")
plt.show()

fig3 = plotter.plot_TddS(concentration_type="molal", folding=False,
                         exp_conc=exp_conc, exp_TddS=exp_TddS)
fig3.suptitle(r"AQ16+Sucrose — $T\Delta\Delta S^0$")
plt.show()

---
## Task B4 — Evaluate Fit Quality (RMSE)

In [ ]:
# Convert experimental molal concentrations -> volume fraction
exp_phi = np.interp(exp_conc, model.molal, model.phiC)

# Interpolate model ΔΔG⁰ at experimental phi values
model_ddG_at_exp = np.interp(exp_phi, model.phiC, model.ddA_kj)

# RMSE
rmse = np.sqrt(np.mean((model_ddG_at_exp - exp_ddG)**2))
print(f"RMSE(ΔΔG⁰) = {rmse:.4f} kJ/mol")

**Answer:** An RMSE below ~0.1 kJ/mol is typically acceptable for biophysical stability data measured by DSC/CD.
If the RMSE were too large, one could try fitting χ as well, or check whether the SASA is correct.

---
## Task B5 — Uncertainty via Monte Carlo Subsampling

In [ ]:
def fit_eps_fn(exp_conc_sub, exp_ddG_sub):
    m = cr.CrowdingModel(
        protein=cr.Protein(SASA=242.6),
        cosolute=cr.Cosolute(nu=11.9, chi=0.452, chiTS=-0.854),
        eps=0.0, epsTS=0.0,
        phiC_max=0.40, dphiC=0.001
    )
    m.fit_eps(exp_conc_sub, exp_ddG_sub, concentration_type="molal")
    return m.eps

results = cr.monte_carlo_subsampling(
    fit_fn=fit_eps_fn,
    exp_conc=exp_conc,
    exp_values=exp_ddG,
    n_iter=50,
    fraction=0.80
)

eps_values = results["fitted_value"].values
print(f"eps = {eps_values.mean():.4f} ± {eps_values.std():.4f} kJ/mol (mean ± std, n=50)")

fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(eps_values, bins=12, edgecolor="white")
ax.set_xlabel("Fitted eps [kJ/mol]")
ax.set_ylabel("Count")
ax.set_title("Monte Carlo distribution of eps")
plt.tight_layout()
plt.show()